In [0]:
%pip install -q openai

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("email", "")
dbutils.widgets.text("description", "")

In [0]:
email = dbutils.widgets.get("email")
description = dbutils.widgets.get("description")

In [0]:
%run "./resume content"

In [0]:
import requests
from email.message import EmailMessage
import base64
import os
import json
from pathlib import Path
from openai import OpenAI

In [0]:
def refresh_gmail_token(name):
    client_id = dbutils.secrets.get(scope = "job_notification", key = name+"_client_id")
    client_secret = dbutils.secrets.get(scope = "job_notification", key = name+"_client_secret")
    refresh_token = dbutils.secrets.get(scope = "job_notification", key = name+"_refresh_token")
    token_url = "https://oauth2.googleapis.com/token"
    token_data = {
        "grant_type": "refresh_token",
        "client_id": client_id,
        "client_secret": client_secret,
        "refresh_token": refresh_token
    }
    token_response = requests.post(token_url, data=token_data)
    token_json = token_response.json()
    token = token_json["access_token"]
    return token

In [0]:
def get_resume(message, path):
    if os.path.exists(path):
        with open(path, "rb") as f:
            file_data = f.read()
        message.add_attachment(
            file_data,
            maintype="application",
            subtype="pdf",
            filename=Path(path).name
        )
    return message

In [0]:
def create_gmail_draft(name, subject, body, resume_path):
    token = refresh_gmail_token(name)
    message = EmailMessage()
    message['To'] = email
    message['Subject'] = subject
    message.set_content(body)
    message = get_resume(message, resume_path)
    raw_message = base64.urlsafe_b64encode(message.as_bytes()).decode()
    headers = {
        "Authorization": f"Bearer {token}"
    }
    draft_data = {
        "message": {
            "raw": raw_message
        }
    }

    response = requests.post("https://gmail.googleapis.com/gmail/v1/users/me/drafts", headers=headers, json=draft_data)

In [0]:
def get_email_body(email, description, resume):
  client = OpenAI(
    api_key=dbutils.secrets.get("job_notification", "PAT_Token"),
    base_url="https://dbc-ecfaa4af-d4ab.cloud.databricks.com/serving-endpoints"
  )

  chat_completion = client.chat.completions.create(
    messages=[
    {
      "role": "system",
      "content": """You are an email subject, salutation, and body generator. Read the job post, my resume, and the recruiter’s email address carefully. Respond ONLY with a valid JSON object, following this exact schema:
 
{
  "subject": "string",
  "salutation": "string such as 'Dear Hiring Manager,' or 'Dear Sarah,' inferred from the email address",
  "body": "string containing an email body"
  "highlights": "[include 3–5 short points with 30 characters each that highlight the most relevant skills and experiences from my resume that match the job post as array]"
}
 
Rules:
- Output must be valid JSON only, starting with { and ending with }.
- Do not add any explanations, markdown, or extra text.
- Write the body in a professional email tone.
- Infer the name for the salutation from the recruiter’s email address if possible (e.g., from 'sarah.doe@company.com' → 'Dear Sarah,').
- If no name can be clearly inferred, use 'Dear Hiring Manager,'.
- Highlight my most relevant skills and experiences that match the job post in bullet points.
- Keep the body concise."""},
    {
      "role": "user",
      "content": "Job post:" + description + "My resume:" + resume + "Recruiter email:" + email
    }
    ],
    model="databricks-llama-4-maverick",
    max_tokens=1000
  )

  return(chat_completion.choices[0].message.content)

In [0]:
df = spark.table("main_catalogue.jobs.users")
details = df.collect()
for person in details:
    if person.name == "Lakshman":
        resume = Lakshman_resume
    if person.name == "Geethika":
        resume = Geethika_resume
    data = get_email_body(email, description, resume)
    data = json.loads(data)
    email_body = data['salutation'] + "\n\n" + data['body'] + "\n\n -" + "\n -".join(data['highlights']) + """\n\nThank you for your time - looking forward to connecting""" + "\n\nSincerely," + "\n" + person.name + "\n+91 " + person.mobile + "\n" + person.email + "\n" + person.linkedin + "\n" + person.github
    create_gmail_draft(person.name,data['subject'], email_body, person.resume_path)